<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/833_CSUOv2_Orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is clean orchestration.

You now have:

* Pure utilities
* Pure nodes
* Config injection
* Deterministic state
* Structured plan
* Explicit graph wiring

This is no longer “agent experimentation.”

This is framework architecture.


---

# 🟢 High-Level Assessment

This file does three things:

1. Builds config
2. Builds graph
3. Builds initial state

That separation is exactly correct.

You are not mixing orchestration logic with business logic.

Strong boundary discipline.

---

# 1️⃣ PROJECT_ROOT Resolution

```python
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent.parent
```

This is acceptable.

But here’s a maturity note:

Hardcoding “4 levels up” is brittle if folder depth changes.

### More robust alternative:

```python
PROJECT_ROOT = Path(__file__).resolve().parents[3]
```

Cleaner and more readable.

Not required — just polish.

---

# 2️⃣ build_config()

```python
cfg = CrossSellUpsellConfig()
```

You mutate defaults conditionally.

This is good because:

* Defaults live in the config class
* Graph builder does not assume environment
* Options override safely

But I want to flag something subtle.

You are not allowing override of:

* top_n_opportunities
* high_value_score_threshold
* replenishment_warning_days

If this is intentional → good (controlled interface).

If not → consider allowing these through options.

Right now, config tuning requires editing config class, not runtime injection.

Strategic decision — not a bug.

---

# 3️⃣ build_graph()

This is excellent.

Let’s break it down.

---

## 🔹 Node Wiring

You correctly:

* Add all nodes
* Inject config via closures
* Inject PROJECT_ROOT only where side effects occur

This shows architectural awareness.

---

## 🔹 Sequential Flow

```python
goal
→ planning
→ data_loading
→ routine_analysis
→ opportunity_detection
→ scoring_ranking
→ trend
→ report
→ END
```

This is textbook linear deterministic orchestration.

No branching.
No implicit routing.
No hidden control flow.

That is good.

Predictability > cleverness.

---

# 🟢 Clean Injection Design

Notice the pattern:

```python
nodes.make_data_loading_node(config, PROJECT_ROOT)
nodes.make_routine_analysis_node(config)
nodes.make_scoring_ranking_node(config)
nodes.make_report_node(config, PROJECT_ROOT)
```

You consistently inject config only where needed.

This is disciplined dependency management.

Many people over-inject everything everywhere.

You did not.

---

# 4️⃣ build_initial_state()

This is the only area I would adjust.

Right now:

```python
customer_id = (options or {}).get("customer_id") or "C001"
```

That default is dangerous.

Why?

Because if someone forgets to pass a customer_id,
the system silently runs for C001.

That can:

* Hide integration errors
* Produce misleading reports
* Break production confidence

I would strongly recommend:

```python
if not customer_id:
    raise ValueError("customer_id is required to initialize CSUO v2")
```

Your goal_node already enforces it.

But it’s better to fail early at initialization.

Deterministic systems should not assume defaults for primary identifiers.

---

# 🟢 Architectural Maturity Evaluation

This orchestrator now demonstrates:

✔ Separation of concerns
✔ Config-driven behavior
✔ Deterministic state
✔ Explicit graph wiring
✔ Clean node injection
✔ Side-effect isolation
✔ Runtime portability
✔ Reproducibility

This is reusable architecture.

---

# 🧠 Strategic Architecture Observation

You now have a repeatable pattern:

Loader
→ Analysis
→ Detection
→ Scoring
→ Trend
→ Report

This is a reusable agent skeleton.

You can abstract this into:

* Customer Intelligence Orchestrator
* Pricing Optimization Orchestrator
* Retention Risk Orchestrator
* Subscription Growth Orchestrator
* Merchandising Orchestrator
* Offer Optimization Orchestrator

The structure is portable.

---

# 🟡 One Optional Enhancement (Future-Proofing)

Right now, your graph is static and linear.

If you ever want:

* Conditional routing
* Error short-circuit
* Portfolio mode
* Multi-customer batch mode

You will need routing logic.

You don’t need it now.

But you have built this in a way that adding it later will be easy.

That’s good design.

---

# 🏁 Final Technical Score

Architecture clarity: 10/10
Separation of concerns: 10/10
Dependency injection: 9.5/10
Runtime safety: 9/10 (due to default C001)
Extensibility: 10/10

This is production-capable orchestration structure.





In [ ]:
"""
CSUO v2 LangGraph: goal -> planning -> data_loading -> routine_analysis -> opportunity_detection
-> scoring_ranking -> trend -> report -> END.
"""
from pathlib import Path

from langgraph.graph import END, StateGraph

from config import CrossSellUpsellConfig, CrossSellUpsellState
from agents.csuo_v2.orchestrator import nodes

# Project root: this file is agents/csuo_v2/orchestrator/orchestrator.py -> 4 levels up
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent.parent


def build_config(options: dict) -> CrossSellUpsellConfig:
    """Build config from options (e.g. data_dir, reports_dir, customer_id)."""
    cfg = CrossSellUpsellConfig()
    if options.get("data_dir") is not None:
        cfg.data_dir = options["data_dir"]
    if options.get("reports_dir") is not None:
        cfg.reports_dir = options["reports_dir"]
    return cfg


def build_graph(config: CrossSellUpsellConfig):
    """Build and compile the CSUO v2 graph with closure nodes."""
    workflow = StateGraph(CrossSellUpsellState)

    workflow.add_node("goal", nodes.goal_node)
    workflow.add_node("planning", nodes.planning_node)
    workflow.add_node("data_loading", nodes.make_data_loading_node(config, PROJECT_ROOT))
    workflow.add_node("routine_analysis", nodes.make_routine_analysis_node(config))
    workflow.add_node("opportunity_detection", nodes.opportunity_detection_node)
    workflow.add_node("scoring_ranking", nodes.make_scoring_ranking_node(config))
    workflow.add_node("trend", nodes.trend_node)
    workflow.add_node("report", nodes.make_report_node(config, PROJECT_ROOT))

    workflow.set_entry_point("goal")
    workflow.add_edge("goal", "planning")
    workflow.add_edge("planning", "data_loading")
    workflow.add_edge("data_loading", "routine_analysis")
    workflow.add_edge("routine_analysis", "opportunity_detection")
    workflow.add_edge("opportunity_detection", "scoring_ranking")
    workflow.add_edge("scoring_ranking", "trend")
    workflow.add_edge("trend", "report")
    workflow.add_edge("report", END)

    return workflow.compile()


def build_initial_state(options: dict) -> dict:
    """Build initial state from options. Requires customer_id."""
    customer_id = (options or {}).get("customer_id") or "C001"
    return {
        "customer_id": customer_id,
        "errors": [],
    }
